In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

G = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 1))
D = nn.Sequential(nn.Linear(1, 8), nn.ReLU(), nn.Linear(8, 1), nn.Sigmoid())

loss = nn.BCELoss()

opt_G = optim.Adam(G.parameters(), lr=0.01)
opt_D = optim.Adam(D.parameters(), lr=0.01)

for _ in range(50):
    
    real = torch.randn(5, 1)
    fake = G(torch.randn(5, 3))
    
    d_loss = loss(D(real), torch.ones(5,1)) + loss(D(fake.detach()), torch.zeros(5,1))
    opt_D.zero_grad()
    d_loss.backward()
    opt_D.step()
    
    fake = G(torch.randn(5, 3))
    g_loss = loss(D(fake), torch.ones(5,1))
    opt_G.zero_grad()
    g_loss.backward()
    opt_G.step()

print("Generated Data:\n", G(torch.randn(3,3)).detach().numpy())

In [ ]:
import torch
import torch.nn as nn
import torchvision
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms

# Load dataset
data = torchvision.datasets.MNIST(
    "./data", train=True, download=True, transform=transforms.ToTensor()
)
loader = DataLoader(data, batch_size=64, shuffle=True)

# Generator
G = nn.Sequential(
    nn.Linear(100,128),
    nn.ReLU(),
    nn.Linear(128,784),
    nn.Tanh()
)

# Discriminator
D = nn.Sequential(
    nn.Linear(784,128),
    nn.ReLU(),
    nn.Linear(128,1),
    nn.Sigmoid()
)

loss = nn.BCELoss()
optG = torch.optim.Adam(G.parameters(),0.0002)
optD = torch.optim.Adam(D.parameters(),0.0002)

# Training
for epoch in range(5):
    for real,_ in loader:

        real = real.view(-1,784)
        bs = real.size(0)

        real_label = torch.ones(bs,1)
        fake_label = torch.zeros(bs,1)

        # Train Discriminator
        z = torch.randn(bs,100)
        fake = G(z)

        lossD = loss(D(real),real_label) + loss(D(fake.detach()),fake_label)
        optD.zero_grad(); lossD.backward(); optD.step()

        # Train Generator
        lossG = loss(D(fake),real_label)
        optG.zero_grad(); lossG.backward(); optG.step()

    print("Epoch:",epoch+1,"LossD:",lossD.item(),"LossG:",lossG.item())

# Generate images
z = torch.randn(16,100)
images = G(z).view(-1,28,28).detach()

for i in range(16):
    plt.subplot(4,4,i+1)
    plt.imshow(images[i], cmap="gray")
    plt.axis("off")

plt.show()

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"

z_dim, batch_size, epochs = 100, 128, 30

transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,), (0.5,))])

loader = DataLoader(
    datasets.MNIST("./data", train=True, download=True, transform=transform),
    batch_size=batch_size, shuffle=True)

# Generator
G = nn.Sequential(
    nn.Linear(z_dim,256), nn.LeakyReLU(0.2),
    nn.Linear(256,512), nn.LeakyReLU(0.2),
    nn.Linear(512,1024), nn.LeakyReLU(0.2),
    nn.Linear(1024,784), nn.Tanh()
).to(device)

# Discriminator
D = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784,512), nn.LeakyReLU(0.2),
    nn.Linear(512,256), nn.LeakyReLU(0.2),
    nn.Linear(256,1), nn.Sigmoid()
).to(device)

loss = nn.BCELoss()
optG = optim.Adam(G.parameters(),0.0002,(0.5,0.999))
optD = optim.Adam(D.parameters(),0.0002,(0.5,0.999))

for epoch in range(1,epochs+1):
    for real,_ in loader:
        real = real.to(device)
        bs = real.size(0)

        real_label = torch.ones(bs,1).to(device)
        fake_label = torch.zeros(bs,1).to(device)

        # Train Discriminator
        z = torch.randn(bs,z_dim).to(device)
        fake = G(z).view(-1,1,28,28)

        lossD = loss(D(real),real_label) + loss(D(fake.detach()),fake_label)
        optD.zero_grad(); lossD.backward(); optD.step()

        # Train Generator
        z = torch.randn(bs,z_dim).to(device)
        fake = G(z).view(-1,1,28,28)

        lossG = loss(D(fake),real_label)
        optG.zero_grad(); lossG.backward(); optG.step()

    print(f"Epoch {epoch}/{epochs}  Loss_D:{lossD.item():.4f}  Loss_G:{lossG.item():.4f}")

    # Show images every 5 epochs
    if epoch % 5 == 0:
        with torch.no_grad():
            z = torch.randn(16,z_dim).to(device)
            samples = G(z).view(-1,1,28,28).cpu()*0.5 + 0.5

        fig,ax = plt.subplots(1,16,figsize=(16,2))
        for i in range(16):
            ax[i].imshow(samples[i][0],cmap="gray")
            ax[i].axis("off")
        plt.title(f"Epoch {epoch}")
        plt.show()